In [1]:
import pandas as pd
import numpy as np

# Read selected raw variables
df_project = pd.read_csv(
    "../data/data_processed/df_project_raw.csv",
    low_memory=False
)

print("Original shape:", df_project.shape)

Original shape: (15985, 58)


In [2]:
# 1. Process outcome variables

# Feasibility of self-isolation

feasibility_map = {
    "Very difficult": 1,
    "Somewhat difficult": 2,
    "Neither easy nor difficult": 3,
    "Somewhat easy": 4,
    "Very easy": 5
}

df_project["feasibility_score"] = df_project["i10_health"].map(
    feasibility_map
)

df_project["feasibility_binary"] = np.where(
    df_project["feasibility_score"].notna(),
    (df_project["feasibility_score"] >= 4).astype(int),
    np.nan
)


# Willingness to self-isolate

willingness_map = {
    "Very unwilling": 1,
    "Somewhat unwilling": 2,
    "Neither willing nor unwilling": 3,
    "Somewhat willing": 4,
    "Very willing": 5
}

df_project["willingness_score"] = df_project["i11_health"].map(
    willingness_map
)

df_project["willingness_binary"] = np.where(
    df_project["willingness_score"].notna(),
    (df_project["willingness_score"] >= 4).astype(int),
    np.nan
)


# 2. Check outcomes before removing missing responses
for var in [
    "feasibility_score",
    "feasibility_binary",
    "willingness_score",
    "willingness_binary"
]:
    print(f"\n{var}")
    print(
        df_project[var]
        .value_counts(dropna=False)
        .sort_index()
        .rename_axis(None)
        .rename(None)
    )


# 3. Keep the common analysis sample
df_project = df_project.dropna(
    subset=["feasibility_binary", "willingness_binary"]
).copy()

print("\nShape after keeping valid responses for both outcomes:")
print(df_project.shape)


feasibility_score
1.0     152
2.0     874
3.0    2431
4.0    5963
5.0    5130
NaN    1435
dtype: int64

feasibility_binary
0.0     3457
1.0    11093
NaN     1435
dtype: int64

willingness_score
1.0     121
2.0     465
3.0    1700
4.0    5515
5.0    6810
NaN    1374
dtype: int64

willingness_binary
0.0     2286
1.0    12325
NaN     1374
dtype: int64

Shape after keeping valid responses for both outcomes:
(14512, 62)


In [3]:
# Save
df_project.to_csv(
    "../data/data_processed/df_outcomes_processed.csv",
    index=False
)